In [0]:
from pyspark.sql import functions as F
from delta import DeltaTable
from pyspark.sql.window  import Window

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
df_bronze=spark.read\
    .option("header",True)\
    .option("inferSchema",True)\
    .csv("s3://sportsbar-dp-sh/gross_price/*.csv")

dbutils.widgets.text("catalog","fmcg")
dbutils.widgets.text("data_source","gross_price")

catalog=dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")

df_bronze=(df_bronze.withColumn("ingest_time",F.current_timestamp())
           .withColumn("file_name",F.col("_metadata.file_name"))
           .withColumn("file_path",F.col("_metadata.file_path"))
           .withColumn("file_size",F.col("_metadata.file_size")))

df_bronze.write.format("delta")\
    .mode("overwrite")\
    .option("mergeschema",True)\
    .option("header",True)\
    .option("delta.enableChangeDataFeed",True)\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")


In [0]:
df_silver=spark.table(f"{catalog}.{bronze_schema}.{data_source}")
df_silver=df_silver.withColumn("gross_price",F.abs(F.col("gross_price").try_cast("double")))
df_silver=df_silver.withColumn("gross_price",F.when(F.col("gross_price")>0,F.col("gross_price")).otherwise(0))
df_silver = df_silver.withColumn("month",
    F.date_format(
        F.coalesce(
            F.try_to_date(F.col("month"), "yyyy/MM/dd"),
            F.try_to_date(F.col("month"), "dd/MM/yyyy"),
            F.try_to_date(F.col("month"), "yyyy-MM-dd"),
            F.try_to_date(F.col("month"), "dd-MM-yyyy")
        ),
        "yyyy-MM-dd"
    )
)
df_silver=df_silver.withColumn("year",F.year(F.col("month")))

df_products = spark.table(f"{catalog}.{silver_schema}.products") 
df_joined = df_silver.join(df_products.select("product_id", "product_code"), on="product_id", how="inner")
df_joined = df_joined.select("product_id", "product_code", "month", "gross_price", "ingest_time", "file_name", "file_size")

In [0]:
# Recreate df_joined since it was set to None by .show() in Cell 5
df_silver=spark.table(f"{catalog}.{bronze_schema}.{data_source}")
df_silver=df_silver.withColumn("gross_price",F.abs(F.col("gross_price").try_cast("double")))
df_silver=df_silver.withColumn("gross_price",F.when(F.col("gross_price")>0,F.col("gross_price")).otherwise(0))
df_silver = df_silver.withColumn("month",
    F.date_format(
        F.coalesce(
            F.try_to_date(F.col("month"), "yyyy/MM/dd"),
            F.try_to_date(F.col("month"), "dd/MM/yyyy"),
            F.try_to_date(F.col("month"), "yyyy-MM-dd"),
            F.try_to_date(F.col("month"), "dd-MM-yyyy")
        ),
        "yyyy-MM-dd"
    )
)
df_silver=df_silver.withColumn("year",F.year(F.col("month")))

df_products = spark.table(f"{catalog}.{silver_schema}.products") 
df_joined = df_silver.join(df_products.select("product_id", "product_code"), on="product_id", how="inner")
df_joined = df_joined.select("product_id", "product_code", "month", "gross_price", "ingest_time", "file_name", "file_size")

df_joined.write.format("delta")\
    .option("header",True)\
    .mode("overwrite")\
    .option("mergeSchema",True)\
    .option("delta.enableChangeDataFeed",True)\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")


**Gold Layer Processing**

In [0]:
df_silver=spark.table(f"{catalog}.{silver_schema}.{data_source}")

df_gold=df_silver.select(F.col("product_code"),F.col("month"),F.col("gross_price"))

df_gold.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema",True)\
    .option("delta.enableChangeDataFeed",True)\
    .option("header",True)\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")
    



In [0]:
df_gold=spark.table(f"{catalog}.{gold_schema}.sb_dim_gross_price")

df_gold=(df_gold.withColumn("year",F.year(F.col("month")))
         .withColumn("is_zero",F.when(F.col("gross_price")==0,1)
                     .otherwise(0)))
w=Window.partitionBy("product_code","year").orderBy(F.col("gross_price").desc())

df_gold=df_gold.withColumn("rnk", F.row_number().over(w))
df_gold=df_gold.filter(F.col("rnk")==1)

In [0]:

## Take required cols

df_gold=df_gold.select("product_code","year","gross_price")
df_gold=(df_gold.withColumnRenamed("gross_price","price_inr")
         .withColumn("year",F.col("year").cast("string")))


delta_table_pro=DeltaTable.forName(spark,"fmcg.gold.dim_gross_price")

delta_table_pro.alias("target").merge(
   source= df_gold.alias("source"),
   condition= "target.product_code = source.product_code"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()
